# 01 – Sentiment-baseline for russiske tekster

Denne notebooken er et **hobbyprosjekt** og brukes kun til **egen læring og utforsking**.  
Målet er å bygge en første, enkel baseline for sentimentanalyse på russiskspråklige tekster.

Jeg ønsker å:

- øve på praktisk NLP med ekte data
- bli tryggere på arbeidsflyten:
  - hente datasett
  - forberede tekstdata
  - trene og evaluere en modell
- ha et konkret utgangspunkt jeg senere kan sammenligne mer avanserte modeller med  
  (f.eks. BERT-baserte modeller for russisk)

Prosjektet er ikke laget for produksjon, men for å:

- forstå styrker og svakheter ved en helt enkel modell
- bygge opp en portefølje av små, gjennomførte analyser

---

## Hva denne notebooken gjør

I grove trekk gjør notebooken følgende:

1. **Laster inn et annotert russisk sentiment-datasett**  
   – via `datasets`-biblioteket (Hugging Face), f.eks. RuSentiment eller et lignende korpus.

2. **Utforsker datasettet kort**  
   - ser på noen eksempler (tekst + label)  
   - undersøker fordelingen av sentimentklasser (f.eks. positiv, negativ, nøytral)

3. **Forbereder data for modelltrening**  
   - deler i trenings- og testsett (eller bruker ferdige splits)  
   - tilpasser tekst og labels til formatet som `scikit-learn` forventer

4. **Bygger en enkel baseline-modell**  
   - representerer tekst som TF–IDF-vektorer (ord og eventuelt bigrams)  
   - trener en logistisk regresjon som predikerer sentiment basert på disse vektorene

5. **Evaluerer modellen**  
   - måler:
     - nøyaktighet (accuracy)
     - precision, recall og $F_1$-score per klasse og som gjennomsnitt  
   - ser kort på hvilke klasser modellen håndterer bedre/dårligere

6. **Oppsummerer resultatene**  
   - enkel refleksjon over:
     - hvor “bra” en slik baseline egentlig er
     - hva som kan forbedres i senere eksperimenter (f.eks. bedre features eller dypere modeller)

---

## Avgrensninger og intensjon

- Notebooken er **ikke** ment som et ferdig forskningsprosjekt, men som et **læringsverktøy**.
- Koden og analysene er skrevet for å være:
  - forståelige for meg selv i etterkant
  - en del av en offentlig portefølje som viser hvordan jeg jobber
- Datasettet og modellene brukes utelukkende til:
  - å undersøke språklige mønstre og sentiment i russiske tekster
  - å lære mer om praktisk NLP og maskinlæring

Senere vil jeg kunne:

- forbedre denne baselinen
- sammenligne med mer avanserte modeller
- koble resultatene mot større mål i prosjektet (f.eks. analyse av propaganda, mediebias og retorikk om krigen i Ukraina).


In [2]:
# Grunnleggende pakker
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd

import sys
import platform


In [ ]:
# Laster "k1tub/sentiment_dataset" fra Hugging Face
dataset = load_dataset("k1tub/sentiment_dataset")

dataset


c:\Users\nibre\OneDrive\Dokumenter\russian-media-ukraine-analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nibre\.cache\huggingface\hub\datasets--k1tub--sentiment_dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 290458/290458 [00:00<00:00, 12

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 290458
    })
})

In [7]:
# Første eksempel
example = dataset["train"][0]
example

{'text': 'Пальто красивое, но пришло с дырой в молнии. Просила выслать такую же, но продавец настаивал на открытии спора.\r\nАли экспресс предложил либо вернуть товар либо компенсацию 659р.При возврате непонятно, кто оплачивает доставку, к тому же я хотела носить пальто. В ателье выяснилось, что вся молния обрезана, а сверху защита тонкой планкой.',
 'label': 0,
 'src': 'rureviews'}

In [8]:
# Kolonnenavn
dataset["train"].column_names


['text', 'label', 'src']

In [9]:
# Unike label-verdier
unique_labels = set(dataset["train"]["label"])
unique_labels


{0, 1, 2}